# Step Overrides

## What you'll learn

- Override operation parameters, names, and execution config per step
- Control failure handling with `FailurePolicy`
- Understand override precedence (pipeline defaults → operation defaults → step overrides)

**Prerequisites:** [First Pipeline](../01-getting-started/01-first-pipeline.ipynb),
[Batching and Performance](../04-batching/01-batching-and-performance.ipynb).
**Estimated time:** 15 minutes
**GPU required:** No.

---

Every `pipeline.run()` call accepts optional overrides that customize how the
step executes. This tutorial demonstrates each override parameter with working
examples.


In [ ]:
from __future__ import annotations

from artisan.operations.examples import (
    DataGenerator,
    DataTransformer,
    MetricCalculator,
)
from artisan.orchestration import PipelineManager, Runner
from artisan.schemas.enums import FailurePolicy, GroupByStrategy
from artisan.utils import tutorial_setup
from artisan.visualization import build_macro_graph, build_micro_graph, inspect_pipeline

In [ ]:
env = tutorial_setup("step_overrides")

## Override parameters

Each `pipeline.run()` call accepts override parameters for batching,
resources, and the step runner. For the complete list, see
[Configuring Execution](../../how-to-guides/configuring-execution.md).

The most commonly used overrides are `params`, `name`, `batch_strategy`,
`step_runner`, `runner_resources`, `failure_policy`, and `group_by`. This tutorial demonstrates
each one with working examples.


## `params` — operation parameters

The `params` dict is passed directly to the operation. Each operation defines
its own parameter schema. Passing different params to the same operation
creates distinct steps with different behavior.


In [ ]:
pipeline = PipelineManager.create(
    name="params_demo",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)

# Same operation, different params → different outputs
pipeline.run(
    operation=DataGenerator,
    name="small",
    params={"count": 3, "seed": 42},
    step_runner=Runner.LOCAL,
)
pipeline.run(
    operation=DataGenerator,
    name="large",
    params={"count": 10, "seed": 99},
    step_runner=Runner.LOCAL,
)

pipeline.finalize()
inspect_pipeline(env.delta_root)

Both steps use `DataGenerator` but produce different outputs: step 0 generates
3 datasets with seed 42, step 1 generates 10 with seed 99. The pipeline
overview shows the different artifact counts.


## `name` — custom step name

By default, each step is named after the operation (e.g., `data_generator`).
Use `name` to give steps descriptive labels, especially when the same
operation appears multiple times.


In [ ]:
env_name = tutorial_setup("step_names")

pipeline = PipelineManager.create(
    name="name_demo",
    delta_root=env_name.delta_root,
    staging_root=env_name.staging_root,
    working_root=env_name.working_root,
)
output = pipeline.output

pipeline.run(
    operation=DataGenerator,
    name="initial_candidates",
    params={"count": 5, "seed": 42},
    step_runner=Runner.LOCAL,
)
pipeline.run(
    operation=DataTransformer,
    name="normalize",
    inputs={"dataset": output("initial_candidates", "datasets")},
    step_runner=Runner.LOCAL,
)
pipeline.run(
    operation=DataTransformer,
    name="augment",
    inputs={"dataset": output("normalize", "dataset")},
    params={"seed": 100},
    step_runner=Runner.LOCAL,
)

pipeline.finalize()
inspect_pipeline(env_name.delta_root)

The pipeline overview now shows `initial_candidates`, `normalize`, and
`augment` instead of generic operation names. Custom names make pipelines
easier to read, especially in provenance graphs.


In [ ]:
build_macro_graph(env_name.delta_root)

## `batch_strategy` — batching configuration

The `batch_strategy` dict accepts any `BatchStrategy` field. See
[Batching and Performance](../04-batching/01-batching-and-performance.ipynb) for a deep dive.


In [ ]:
env_exec = tutorial_setup("step_execution")

pipeline = PipelineManager.create(
    name="execution_demo",
    delta_root=env_exec.delta_root,
    staging_root=env_exec.staging_root,
    working_root=env_exec.working_root,
)
output = pipeline.output

pipeline.run(
    operation=DataGenerator,
    name="generate",
    params={"count": 12, "seed": 42},
    step_runner=Runner.LOCAL,
)

# Override batching: 4 artifacts per unit, max 2 concurrent workers
pipeline.run(
    operation=MetricCalculator,
    name="metrics",
    inputs={"dataset": output("generate", "datasets")},
    batch_strategy={"artifacts_per_unit": 4, "max_workers": 2},
    step_runner=Runner.LOCAL,
)

pipeline.finalize()
inspect_pipeline(env_exec.delta_root)

Step 1 groups 12 artifacts into 3 execution units (4 per unit) and runs at
most 2 concurrently. Without the override, MetricCalculator's default
`artifacts_per_unit` of 10,000 would put all 12 artifacts into a single unit.


## `step_runner` — step runner

The `step_runner` parameter overrides the pipeline-level default step runner for a
single step. This allows mixing step runners within one pipeline — for example,
running lightweight steps locally while submitting heavy computation to SLURM.

```python
# Pipeline default is SLURM, but run this step locally
pipeline.run(
    MetricCalculator,
    inputs={"dataset": output("generate", "datasets")},
    step_runner=Runner.LOCAL,  # Override for this step only
)
```

Available step runners: `Runner.LOCAL` (process pool on current machine),
`Runner.SLURM` (submits jobs to a SLURM cluster), and
`Runner.SLURM_INTRA` (distributes work via srun within an existing
SLURM allocation).


## `runner_resources` — SLURM resource allocation

The `runner_resources` dict controls SLURM job resources (CPUs, memory, GPUs, time
limit). These have no effect when running locally. For all RunnerResources
fields, see [Configuring Execution](../../how-to-guides/configuring-execution.md).

```python
pipeline.run(
    MyGPUOperation,
    inputs={"data": datasets},
    runner_resources={"gpus": 1, "memory_gb": 32, "extra": {"partition": "gpu"}},
    step_runner=Runner.SLURM,
)
```


## `failure_policy` — handling failures

By default, the pipeline uses the policy set at creation time (default:
`FailurePolicy.CONTINUE`). Override per-step for fine-grained control.

| Policy                    | Behavior                                                               |
| ------------------------- | ---------------------------------------------------------------------- |
| `FailurePolicy.CONTINUE`  | Log failures, commit successful results, report failures in StepResult |
| `FailurePolicy.FAIL_FAST` | Stop on first failure, raise exception, no commit                      |


In [ ]:
env_fp = tutorial_setup("step_failure_policy")

pipeline = PipelineManager.create(
    name="failure_demo",
    delta_root=env_fp.delta_root,
    staging_root=env_fp.staging_root,
    working_root=env_fp.working_root,
    failure_policy=FailurePolicy.CONTINUE,  # Pipeline default
)
output = pipeline.output

pipeline.run(
    operation=DataGenerator,
    name="generate",
    params={"count": 5, "seed": 42},
    step_runner=Runner.LOCAL,
)

# This step must succeed completely — fail fast on any error
pipeline.run(
    operation=DataTransformer,
    name="transform",
    inputs={"dataset": output("generate", "datasets")},
    failure_policy=FailurePolicy.FAIL_FAST,
    step_runner=Runner.LOCAL,
)

# This step tolerates partial failures — continue processing
pipeline.run(
    operation=MetricCalculator,
    name="metrics",
    inputs={"dataset": output("transform", "dataset")},
    failure_policy=FailurePolicy.CONTINUE,
    step_runner=Runner.LOCAL,
)

pipeline.finalize()
inspect_pipeline(env_fp.delta_root)

Step 1 uses `FAIL_FAST` — if any execution unit fails, the entire step aborts
with an exception. Step 2 uses `CONTINUE` — failures are logged and surviving
results are committed. Use `FAIL_FAST` for critical steps where partial results
are meaningless, and `CONTINUE` for steps where you'd rather keep what
succeeded.


## `group_by` — pairing strategy for multi-input steps

When an operation declares two or more input roles, the framework must
pair artifacts across those roles before dispatching execution units.
The operation author can declare a class-level default on
`OperationDefinition.group_by`; the pipeline builder can override it
per step.

| Strategy                        | Pairing rule                                                    |
| ------------------------------- | --------------------------------------------------------------- |
| `GroupByStrategy.LINEAGE`       | Match artifacts that share a common ancestor (most common)      |
| `GroupByStrategy.CROSS_PRODUCT` | Every artifact in role A paired with every artifact in role B   |
| `GroupByStrategy.ZIP`           | Positional: 1st with 1st, 2nd with 2nd (requires equal lengths) |

See [Multi-Input Operations](../02-pipeline-design/04-multi-input-operations.ipynb)
for the strategy semantics in depth.

The override flows into both the cache key and the runtime instance, so
two runs of the same step with different `group_by` values produce
different `step_spec_id`s — a re-run with a new strategy does not return
a stale cached result.


In [ ]:
# Define a small multi-input op that has no class-level `group_by`, so
# the per-step override is what selects the pairing strategy.
import os
from enum import StrEnum
from pathlib import Path
from typing import Any, ClassVar

from artisan.operations.base.operation_definition import OperationDefinition
from artisan.schemas import ArtifactResult
from artisan.schemas.artifact.data import DataArtifact
from artisan.schemas.specs.input_models import (
    ExecuteInput,
    PostprocessInput,
    PreprocessInput,
)
from artisan.schemas.specs.input_spec import InputSpec
from artisan.schemas.specs.output_spec import OutputSpec


class LabeledPair(OperationDefinition):
    """Concatenate paired input files. No class-level group_by."""

    name: ClassVar[str] = "labeled_pair"

    class InputRole(StrEnum):
        left = "left"
        right = "right"

    class OutputRole(StrEnum):
        pair = "pair"

    inputs: ClassVar[dict[str, InputSpec]] = {
        InputRole.left: InputSpec(artifact_type="data"),
        InputRole.right: InputSpec(artifact_type="data"),
    }
    outputs: ClassVar[dict[str, OutputSpec]] = {
        OutputRole.pair: OutputSpec(
            artifact_type="data",
            infer_lineage_from={"inputs": ["left", "right"]},
        ),
    }

    def preprocess(self, inputs: PreprocessInput) -> dict[str, Any]:
        return {
            role: [str(a.materialized_path) for a in artifacts]
            for role, artifacts in inputs.input_artifacts.items()
        }

    def execute(self, inputs: ExecuteInput) -> dict[str, Any]:
        os.makedirs(inputs.execute_dir, exist_ok=True)
        left = Path(inputs.inputs["left"][0])
        right = Path(inputs.inputs["right"][0])
        out = Path(inputs.execute_dir) / f"{left.stem}__{right.stem}.csv"
        # Output bytes depend on BOTH inputs so each pair gets a distinct
        # artifact_id under content-addressed storage.
        out.write_bytes(left.read_bytes() + b"\n---\n" + right.read_bytes())
        return {}

    def postprocess(self, inputs: PostprocessInput) -> ArtifactResult:
        drafts = [
            DataArtifact.draft(
                content=Path(f).read_bytes(),
                original_name=os.path.basename(f),
                step_number=inputs.step_number,
            )
            for f in inputs.file_outputs
            if f.endswith(".csv")
        ]
        return ArtifactResult(success=True, artifacts={"pair": drafts})


env_groupby = tutorial_setup("step_group_by")

pipeline = PipelineManager.create(
    name="group_by_demo",
    delta_root=env_groupby.delta_root,
    staging_root=env_groupby.staging_root,
    working_root=env_groupby.working_root,
)
output = pipeline.output

# Two independent DataGenerator runs — no shared ancestry between them,
# so LINEAGE pairing would match nothing.
pipeline.run(
    operation=DataGenerator,
    name="gen_left",
    params={"count": 2, "seed": 42},
    step_runner=Runner.LOCAL,
)
pipeline.run(
    operation=DataGenerator,
    name="gen_right",
    params={"count": 3, "seed": 100},
    step_runner=Runner.LOCAL,
)

# 2 left x 3 right = 6 paired outputs via per-step CROSS_PRODUCT.
pipeline.run(
    operation=LabeledPair,
    name="pair_all",
    inputs={
        "left": output("gen_left", "datasets"),
        "right": output("gen_right", "datasets"),
    },
    group_by=GroupByStrategy.CROSS_PRODUCT,
    step_runner=Runner.LOCAL,
)

pipeline.finalize()
inspect_pipeline(env_groupby.delta_root)

In [ ]:
build_macro_graph(env_groupby.delta_root)

In [ ]:
build_micro_graph(env_groupby.delta_root)

The `pair_all` step produced 6 outputs — every left dataset paired with
every right dataset. `LabeledPair` declares no class-level strategy, so
the per-step `group_by=GroupByStrategy.CROSS_PRODUCT` is what selects
pairing for this step.

Use `group_by` overrides when an operation's declared strategy doesn't
match a specific step's needs — for example, sweeping across inputs from
independent upstream branches that share no provenance.


## `compute_provider` — compute routing target

The `compute_provider` parameter selects where the execute() phase runs.
This is orthogonal to `step_runner` (which controls where the worker
runs). Pass a string to select a configured provider, or a dict
for inline overrides.

| Provider  | What it does                            |
| --------- | --------------------------------------- |
| `"local"` | Direct call (default)                   |
| `"modal"` | Route to a Modal container (GPU, cloud) |

```python
# Route execute() to Modal for this step
pipeline.run(
    HeavyOp,
    inputs={"data": datasets},
    compute_provider="modal",
)

# Dict syntax for inline config overrides
pipeline.run(
    HeavyOp,
    inputs={"data": datasets},
    compute_provider={"active": "modal", "modal": {"gpu": "A100"}},
)
```

Since compute routing may require provider credentials, this tutorial
does not include a runnable example. See
[Compute Routing](../07-compute-backends/01-compute-routing.ipynb) for a hands-on walkthrough.


## `environment` — container and environment configuration

The `environment` parameter selects which execution environment to use for a
step. Pass a string to select a pre-configured environment, or a dict to
override environment settings.

Available environments: `"local"`, `"docker"`, `"apptainer"`, `"pixi"`.

```python
# Run this step in a Docker container
pipeline.run(
    MyOperation,
    inputs={"data": datasets},
    environment={
        "active": "docker",
        "docker": {"image": "my_image:latest"},
    },
)
```

Since environment overrides require specific container or environment
setups, this tutorial does not include a runnable example.

## `tool` — external tool overrides

The `tool` dict overrides the executable, interpreter, or subcommand for
operations that wrap external programs.

| Field         | Type  | Description                              |
| ------------- | ----- | ---------------------------------------- |
| `executable`  | `str` | Path or name of the binary/script        |
| `interpreter` | `str` | Interpreter prefix (e.g. `"python"`)     |
| `subcommand`  | `str` | Subcommand inserted after the executable |

```python
pipeline.run(
    ToolAOp,
    inputs={"data": datasets},
    tool={"executable": "/opt/tool_a/bin/tool_a"},
)
```

The operation must already define a `tool` in its class — this override
only changes specific fields.


## Override precedence

When the same setting is specified at multiple levels, step-level overrides
win:

```
Pipeline defaults  →  Operation class defaults  →  Step overrides (wins)
```

For example, if the pipeline default failure policy is `CONTINUE`, the
operation class has no override, and a step specifies `FAIL_FAST`, that step
uses `FAIL_FAST`.

This applies to all override parameters: `step_runner`, `runner_resources`,
`batch_strategy`, `compute_provider`, `environment`, `tool`, `failure_policy`,
and `group_by`.
The `params` parameter has no defaults — it is always explicit per step.


## Summary

| Override           | Default                                   | When to use                                                                                                       |
| ------------------ | ----------------------------------------- | ----------------------------------------------------------------------------------------------------------------- |
| `params`           | `{}`                                      | Always — operation-specific configuration                                                                         |
| `name`             | Operation name                            | When the same operation appears multiple times                                                                    |
| `batch_strategy`   | Operation defaults                        | Tuning batching for specific steps                                                                                |
| `step_runner`      | Pipeline default                          | Mixing local and SLURM execution                                                                                  |
| `runner_resources` | Operation defaults                        | SLURM resource tuning (memory, GPUs, time limit)                                                                  |
| `failure_policy`   | Pipeline default                          | Critical steps that must fully succeed                                                                            |
| `group_by`         | Operation class default (often `LINEAGE`) | Multi-input steps where the declared strategy doesn't fit (e.g., CROSS_PRODUCT sweep across independent branches) |
| `environment`      | Operation defaults                        | Container or environment selection                                                                                |
| `tool`             | Operation defaults                        | External tool executable overrides                                                                                |
| `compute_provider` | Pipeline default                          | Routing execute() to remote compute (Modal, cloud)                                                                |
| `compact`          | `True`                                    | Fast back-to-back steps where compaction overhead matters                                                         |

**Key takeaway:** Step overrides give you fine-grained control without
changing operation code. Start with pipeline-level defaults, and override
individual steps only where needed.


## Next steps

- [Batching and Performance](../04-batching/01-batching-and-performance.ipynb) — Deep dive into batch strategy tuning
- [Configuring Execution](../../how-to-guides/configuring-execution.md) — Complete configuration reference
- [Multi-Input Operations](../02-pipeline-design/04-multi-input-operations.ipynb) — Pairing strategy semantics (`LINEAGE`, `CROSS_PRODUCT`, `ZIP`) in depth
- [Error Handling in Practice](02-error-visibility.ipynb) — Runtime failures, failure logs, and FailurePolicy
